<a href="https://colab.research.google.com/github/pandahsu849/Programing_Languages/blob/main/HW4_%E6%96%87%E5%AD%97%E8%B3%87%E6%96%99%E5%B0%8F%E5%88%86%E6%9E%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：PTT → Google Sheet → RAG（整理版）

這份 notebook 保留完整流程：

1. 爬取 PTT movie 文章
2. 寫入指定 Google Sheet
3. 從 Google Sheet 讀回資料
4. 建立 FAISS RAG 索引
5. 用 Gemini 根據 PTT 資料回答問題

主要修正：原本設定了 `SHEET_URL`，但實際用 `gc.open(WORKSHEET_NAME)` 開啟試算表，容易打開錯的 Spreadsheet。新版固定使用 `gc.open_by_url(SHEET_URL)`。


In [ ]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 53.2 MB/s eta 0:00:00


In [ ]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。  
`PTT_WORKSHEET_NAME` 是存放 PTT 原始文章的分頁。


In [ ]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1CQZDzhSqoIbR8JjExDr9eTq0qDzP6vQvUp9JAs4Jmic/edit?usp=sharing"
PTT_WORKSHEET_NAME = "ptt_movie_posts"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]

PTT_MOVIE_INDEX = "https://www.ptt.cc/bbs/movie/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (compatible; Colab PTT crawler)"


## 2. 連線 Google Sheet

這裡是最重要的修正：使用 `open_by_url(SHEET_URL)`，不要用 worksheet 名稱打開 spreadsheet。


In [ ]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


✅ 已開啟試算表：程式語言範例
🔗 https://docs.google.com/spreadsheets/d/1CQZDzhSqoIbR8JjExDr9eTq0qDzP6vQvUp9JAs4Jmic/edit?usp=sharing


In [ ]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
        # 若你要保留舊資料，請先備份 Google Sheet。
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)


ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)
print(f"✅ 已準備 worksheet：{ws_ptt.title}")

✅ 已準備 worksheet：ptt_movie_posts


## 3. PTT movie 爬蟲

這段只負責爬 PTT，不碰 RAG。資料會先存在 `new_posts_df`。


In [ ]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_soup(url):
    resp = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": USER_AGENT},
        cookies=PTT_COOKIES,
    )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆":
        return 100
    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10
    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a:
            continue

        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))
        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")

        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main:
        return "", ""

    # 取出文章建立時間
    created_at = ""
    metalines = main.select("div.article-metaline")
    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    # 移除 meta 與推文
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at


def make_post_id(url):
    # 用文章網址檔名當 post_id，穩定且方便去重
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_movie(pages=2, delay=0.5):
    """爬取 PTT movie 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_MOVIE_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }
                all_rows.append(row)
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")

        prev_url = get_prev_index_url(index_soup)
        if not prev_url:
            break
        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次爬到 {len(df)} 篇文章")
    return df

## 4. 執行爬蟲並寫入 Google Sheet

這一格會：

1. 從 Google Sheet 讀取既有資料
2. 爬取新的 PTT 資料
3. 合併並用 `post_id` 去重
4. 寫回 Google Sheet
5. 再讀一次確認真的寫入成功


In [ ]:
# 你可以調整 pages，例如 pages=1 先測試，確認成功後再改成 3 或 5
# 為了避免被 PTT 阻擋，建議將 delay 增加到 2.0 或 3.0 秒
new_posts_df = crawl_ptt_movie(pages=2, delay=3.0)

old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)

written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")

📄 正在讀取列表頁 1/2: https://www.ptt.cc/bbs/movie/index.html
📄 正在讀取列表頁 2/2: https://www.ptt.cc/bbs/movie/index11000.html
✅ 本次爬到 27 篇文章
📌 Google Sheet 原本有 26 筆
✅ 已寫入 Google Sheet：27 筆
🔍 從 Google Sheet 重新讀回：27 筆
✅ 寫入驗證成功


## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`PTT → Google Sheet → RAG`


In [ ]:
# 從 Google Sheet 重新讀取，作為 RAG 的唯一資料來源
rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)

# 清掉沒有內容的文章
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

rag_source_df.head()


📚 可用於 RAG 的文章數：26


,post_id,title,url,date,author,nrec,created_at,fetched_at,content
0,M.1780291884.A.63D,[新聞]「龍母」談《秘密入侵》《韓索羅》沒人喜歡,https://www.ptt.cc/bbs/movie/M.1780291884.A.63...,06/01,kenny1300175,17,Mon Jun 1 13:31:22 2026,2026-06-01 11:17:24,新聞網址：\nhttps://www.toy-people.com/?p=110606\n原...
1,M.1780289588.A.7D9,[請益] 角頭大橋頭、鬥陣欸的幾個問題,https://www.ptt.cc/bbs/movie/M.1780289588.A.7D...,06/01,ClownT,7,Mon Jun 1 12:53:06 2026,2026-06-01 11:17:22,最近才補完角頭宇宙，還是有幾個問題想請益：\n\n1. 看起來大家都知道大橋頭的麥可在賣藥\...
2,M.1780288734.A.D09,[好雷]屍速禁區 雞掰郎比殭屍可怕,https://www.ptt.cc/bbs/movie/M.1780288734.A.D0...,06/01,GeorgeBear,2,Mon Jun 1 12:38:52 2026,2026-06-01 11:17:19,先講點不雷的讚賞：\n\n以我看過的殭屍韓片，這部可說是數一數二的好。若是把美國也算進來，這...
3,M.1780283480.A.77B,[討論] 曼達洛人與古古 第二周跌幅69%,https://www.ptt.cc/bbs/movie/M.1780283480.A.77...,06/01,labich,41,Mon Jun 1 11:11:17 2026,2026-06-01 11:17:17,https://x.com/i/status/2061095622124396670\n北美...
4,M.1780275866.A.945,[好雷] 後室 近期恐怖佳作,https://www.ptt.cc/bbs/movie/M.1780275866.A.94...,06/01,h2030625,7,Mon Jun 1 09:04:24 2026,2026-06-01 11:17:15,原本第一天看到有幾篇負雷 和脆文標題\n\n覺得完蛋 但是看完覺得還不錯\n\n這部那種詭異...


In [ ]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")


def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))

        text = (f"標題：{title}\n"
                f"作者：{author}\n"
                f"日期：{date}\n"
                f"推文數：{nrec}\n"
                f"內容：{content}")
        docs.append({
            "post_id": str(row.get("post_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding 模型載入完成


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RAG 索引建立完成：26 篇文章，向量維度 384


## 6. Gemini 設定與 RAG 問答

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。


In [ ]:
api_key = userdata.get("gemini")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 若你的帳號不支援這個模型，可改成你可用的 Gemini model name
GEMINI_MODEL_NAME = "gemini-3-flash-preview"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


✅ Gemini 已設定：gemini-3-flash-preview


In [ ]:
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    return response.text

## 7. 快速測試


In [ ]:
question = input("請輸入問題：")
answer = query_rag(question, k=3)
print(answer)


請輸入問題：推薦電影
根據提供的 PTT 資料，為您推薦以下兩部電影：

### 1. 《後室》（Backrooms）
*   **電影特點**：
    *   **視覺與聽覺**：畫面以乾淨、明亮的構圖為主，著重場景拍攝與人物表情特寫。聲音基調安靜，配樂能營造緊張與詭異的氣氛。
    *   **觀影感受**：具備強烈的沉浸感與壓迫感。觀影後會產生「抽離感」，並對空間與聲音變得敏感。對於喜歡探索場景細節（如隱藏道路、小機關）的觀眾來說，能滿足探索欲。
*   **注意事項**：
    *   觀影時建議不要有太多預設，也不建議吃東西。
    *   由於場景具有壓迫感，有身心狀況的人不適合觀看。

### 2. 《計程車司機》（Taxi Driver）
*   **電影特點**：
    *   **製作陣容**：由導演史柯西斯（Martin Scorsese）與演員勞勃·狄尼洛（Robert De Niro）合作的經典之作。
    *   **主題深意**：深刻描繪社會強烈的階級意識、憤世嫉俗的心理以及天真的理想主義。劇情發展虛幻不清，對於男主角心境的刻劃與轉折非常值得再三玩味。
*   **推薦評價**：網友認為該片在 50 年後的今天看來，片中元素依然強烈且令人動容。

---

**參考來源：**
1. [[ 無雷] 後室 沉浸式饗宴] (https://www.ptt.cc/bbs/movie/M.1780216642.A.398.html)
2. [[好雷] 計程車司機] (https://www.ptt.cc/bbs/movie/M.1780311377.A.29A.html)
3. [Re:[無雷]後室 沉浸式饗宴] (https://www.ptt.cc/bbs/movie/M.1780274315.A.B8B.html)


## 8. 建立 Gradio 多頁籤介面

這個區塊將建立一個包含「PTT 爬蟲」和「RAG 問答」兩頁的 Gradio 介面。

- 在「PTT 爬蟲」頁籤，您可以設定爬取頁數和延遲時間，點擊按鈕後會執行爬取並更新 RAG 索引。
- 在「RAG 問答」頁籤，您可以輸入問題並獲得基於 PTT 資料的回答。


In [ ]:
import gradio as gr

# 確保這些變數在全域範圍內可被修改
rag_documents = []
rag_index = None
rag_embeddings = None

def _update_rag_index_from_sheet():
    """從 Google Sheet 重新讀取資料並重建 RAG 索引。"""
    global rag_documents, rag_index, rag_embeddings

    current_rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)
    current_rag_source_df = current_rag_source_df[current_rag_source_df["content"].astype(str).str.strip() != ""].copy()

    if current_rag_source_df.empty:
        rag_documents = []
        rag_index = None
        rag_embeddings = None
        return "RAG 索引：Google Sheet 中無有效文章內容，索引已清空。"

    rag_documents = build_rag_documents(current_rag_source_df)
    rag_index, rag_embeddings = build_faiss_index(rag_documents)
    return f"RAG 索引：已更新，包含 {len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}。"


def run_ptt_crawler_and_update_rag(pages, delay):
    """
    執行 PTT 爬蟲，將資料寫入 Google Sheet，然後更新 RAG 索引。
    """
    status_messages = []

    # 1. 執行爬蟲
    new_posts_df = crawl_ptt_movie(pages=pages, delay=delay)
    status_messages.append(f"✅ 本次爬到 {len(new_posts_df)} 篇文章。\n")

    # 2. 從 Google Sheet 讀取既有資料
    old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
    status_messages.append(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆資料。\n")

    # 3. 合併並去重
    ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
    ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
    ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)
    status_messages.append(f"📝 合併去重後，將寫入 {len(ptt_posts_df)} 筆資料到 Google Sheet。\n")

    # 4. 寫回 Google Sheet
    written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
    status_messages.append(f"✅ 已寫入 Google Sheet：{written_count} 筆。\n")

    # 5. 更新 RAG 索引
    rag_update_status = _update_rag_index_from_sheet()
    status_messages.append(rag_update_status)

    return "".join(status_messages)

def get_articles_for_display():
    """從 Google Sheet 讀取文章，並回傳用於 Gradio Dataframe 的 DataFrame。"""
    df = read_sheet_df(ws_ptt, PTT_HEADER)
    # 選擇需要顯示的欄位，並重新命名以提高可讀性
    display_df = df[['title', 'author', 'date', 'nrec', 'url']].copy()
    display_df.columns = ['標題', '作者', '日期', '推文數', '網址']
    return display_df.head(100) # 預設顯示前100條

# 首次啟動時，先建立或更新 RAG 索引
print("初始化 RAG 索引...")
initial_rag_status = _update_rag_index_from_sheet()
print(initial_rag_status)

初始化 RAG 索引...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RAG 索引：已更新，包含 27 篇文章，向量維度 384。


In [ ]:
def gradio_query_rag(question_input):
    """Gradio 介面調用 query_rag 函數的包裝器。"""
    # 這裡的 k=3 可以根據需求調整，或者讓用戶在 Gradio 介面中設定
    if rag_index is None or not rag_documents:
        return "RAG 索引尚未建立或沒有資料，請先執行「PTT 爬蟲」頁籤爬取資料。"
    return query_rag(question_input, k=3)

with gr.Blocks(theme=gr.themes.Soft(), title="PTT 電影版 RAG 應用") as demo:
    gr.Markdown("# PTT 電影版 RAG 問答系統 🤖")

    with gr.Tab("PTT 爬蟲"): # 第一頁籤
        gr.Markdown("## 爬取 PTT 文章並更新 RAG 資料")
        with gr.Row():
            crawler_pages = gr.Slider(minimum=1, maximum=10, value=2, step=1, label="爬取頁數 (pages)", interactive=True)
            crawler_delay = gr.Slider(minimum=1.0, maximum=10.0, value=3.0, step=0.5, label="每次請求延遲 (秒)", interactive=True)

        crawl_button = gr.Button("開始爬取 PTT 文章並更新 RAG 資料", variant="primary")
        crawl_status_output = gr.Textbox(label="爬蟲狀態", lines=5, interactive=False, scale=1)

        crawl_button.click(
            fn=run_ptt_crawler_and_update_rag,
            inputs=[crawler_pages, crawler_delay],
            outputs=crawl_status_output
        )

    with gr.Tab("瀏覽文章"): # 第二頁籤
        gr.Markdown("## 查看已爬取的 PTT 文章")
        refresh_articles_button = gr.Button("重新載入文章列表", variant="secondary")
        articles_display = gr.Dataframe(
            headers=['標題', '作者', '日期', '推文數', '網址'], # 自訂表頭
            interactive=False, # 讓 Dataframe 不可編輯
            col_count=(5, "fixed"), # 顯示 5 欄，固定欄數
            wrap=True, # 允許文字換行
            # height=400, # 移除 height 參數，避免 TypeError
            scale=1 # 讓 Dataframe 佔滿寬度
        )
        refresh_articles_button.click(
            fn=get_articles_for_display,
            outputs=articles_display
        )
        # 介面首次載入時也載入文章
        demo.load(get_articles_for_display, None, articles_display)


    with gr.Tab("RAG 問答"): # 第三頁籤
        gr.Markdown("## 輸入問題，獲取 PTT 電影版相關回答")
        rag_input = gr.Textbox(
            label="你的問題",
            lines=2,
            placeholder="請輸入你想問的 PTT 電影相關問題...",
            scale=1 # 讓輸入框佔滿寬度
        )
        rag_output = gr.Textbox(
            label="RAG 回答",
            lines=15, # 增加行數以加大輸出介面
            interactive=False, # 輸出框不可編輯
            scale=1 # 讓輸出框佔滿寬度
        )
        rag_button = gr.Button("提問", variant="primary")

        rag_button.click(
            fn=gradio_query_rag,
            inputs=rag_input,
            outputs=rag_output
        )

# 啟動 Gradio 應用
demo.launch(debug=True, share=True)

/tmp/ipykernel_4627/3945359792.py:8: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="PTT 電影版 RAG 應用") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b3981c4c5e72ca5254.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://ed15d26735d3d0a899.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://b3981c4c5e72ca5254.gradio.live


## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet，而不是記憶體中的暫存變數。
